Play Video

In [21]:
import cv2
from ultralytics import YOLO

# Load model hasil training
model = YOLO(r"C:\Users\EagleEyes\belajar\model\best94.pt")
# Buka file video (ganti 'video.mp4' dengan path file kamu)
cap = cv2.VideoCapture(r'C:\Users\EagleEyes\dataset\video\jpo_lenteng_agung2.mp4')

# Jika ingin membuka webcam, gunakan index 0
# cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()  # ret = True jika frame berhasil dibaca
    if not ret:
        break

    # Tampilkan frame
    cv2.imshow('Video', frame)

    # Tekan 'q' untuk keluar
    if cv2.waitKey(25) & 0xFF == ord('q'):
        break

# Lepaskan resource
cap.release()
cv2.destroyAllWindows()

Play Video + Label

In [22]:
import cv2
from ultralytics import YOLO

# Load model hasil training
model = YOLO(r"C:\Users\EagleEyes\belajar\model\best94.pt")

# Buka file video
cap = cv2.VideoCapture(r'C:\Users\EagleEyes\dataset\video\jpo_lenteng_agung2.mp4')

# Daftar nama class sesuai training
class_names = [
    'car', 
    'driver_buckled', 
    'driver_unbuckled', 
    'driver_unknown', 
    'kaca',
    'motor_1_helmet', 
    'motor_1_nohelmet', 
    'motor_2_helmet', 
    'motor_2_nohelmet',
    'motor_more_2', 
    'passanger_buckled', 
    'passanger_unbuckled', 
    'passanger_unknown',
    'plat_nomor'
]

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Jalankan deteksi YOLOv8
    results = model(frame, verbose=False)

    for r in results:
        boxes = r.boxes

        for box in boxes:
            # Ambil class id
            cls = int(box.cls[0])

            # Kalau class id tidak ada di daftar → skip
            if cls < 0 or cls >= len(class_names):
                continue  

            # Ambil koordinat box
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            # Confidence score
            conf = float(box.conf[0])

            # Nama class
            label = f"{class_names[cls]} {conf:.2f}"

            # Gambar kotak
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            # Tulis label
            cv2.putText(frame, label, (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    # Tampilkan hasil
    cv2.imshow("Video", frame)

    if cv2.waitKey(25) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()


In [23]:
import cv2
from ultralytics import YOLO

# Load model hasil training
model = YOLO(r"C:\Users\EagleEyes\belajar\model\best94.pt")

# Buka file video
cap = cv2.VideoCapture(r'C:\Users\EagleEyes\dataset\video\jpo_lenteng_agung2.mp4')

# Daftar nama class sesuai training
class_names = [
    'car', 
    'driver_buckled', 
    'driver_unbuckled', 
    'driver_unknown', 
    'kaca',
    'motor_1_helmet', 
    'motor_1_nohelmet', 
    'motor_2_helmet', 
    'motor_2_nohelmet',
    'motor_more_2', 
    'passanger_buckled', 
    'passanger_unbuckled', 
    'passanger_unknown',
    'plat_nomor'
]

frame_count = 0
results = None  # simpan hasil terakhir

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Hitung frame
    frame_count += 1

    # Jalankan deteksi hanya tiap 20 frame
    if frame_count % 20 == 0:
        results = model(frame, verbose=False)

    # Kalau ada hasil deteksi, gambar box
    if results is not None:
        for r in results:
            boxes = r.boxes
            for box in boxes:
                cls = int(box.cls[0])

                # Skip kalau class ID tidak ada di daftar
                if cls < 0 or cls >= len(class_names):
                    continue

                # Ambil koordinat box
                x1, y1, x2, y2 = map(int, box.xyxy[0])

                # Confidence
                conf = float(box.conf[0])

                # Label
                label = f"{class_names[cls]} {conf:.2f}"

                # Gambar box
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(frame, label, (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    # Tampilkan hasil
    cv2.imshow("Video", frame)

    # Tekan 'q' untuk keluar
    if cv2.waitKey(25) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()


Jalankan deteksi setiap 20 frame

In [37]:
import cv2
import os
from ultralytics import YOLO

# Load model hasil training
model = YOLO(r"C:\Users\EagleEyes\belajar\model\best94.pt")

# Buka file video
cap = cv2.VideoCapture(r'C:\Users\EagleEyes\dataset\video\jpo_lenteng_agung2.mp4')

# Daftar nama class sesuai training
class_names = [
    'car', 
    'driver_buckled', 
    'driver_unbuckled', 
    'driver_unknown', 
    'kaca',
    'motor_1_helmet', 
    'motor_1_nohelmet', 
    'motor_2_helmet', 
    'motor_2_nohelmet',
    'motor_more_2', 
    'passanger_buckled', 
    'passanger_unbuckled', 
    'passanger_unknown',
    'plat_nomor'
]

# Buat direktori output
output_dir = "detections"
os.makedirs(output_dir, exist_ok=True)

frame_count = 0
save_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    # Jalankan deteksi setiap 20 frame
    if frame_count % 20 == 0:
        results = model(frame, verbose=False)

        for r in results:
            boxes = r.boxes
            if len(boxes) > 0:
                for box in boxes:
                    cls = int(box.cls[0])
                    conf = float(box.conf[0])
                    
                    # Pastikan index valid
                    if cls < 0 or cls >= len(class_names):
                        continue

                    # Koordinat bounding box
                    x1, y1, x2, y2 = map(int, box.xyxy[0])

                    # Crop objek dari frame
                    cropped = frame[y1:y2, x1:x2]

                    if cropped.size == 0:
                        continue  # skip kalau crop kosong

                    # Simpan dengan nama sesuai class
                    save_count += 1
                    class_name = class_names[cls]
                    save_path = os.path.join(output_dir, f"{class_name}_{save_count:04d}.jpg")
                    cv2.imwrite(save_path, cropped)
                    print(f"Objek {class_name} disimpan ke {save_path}")

    # Tampilkan video dengan box
    for r in results:
        for box in r.boxes:
            cls = int(box.cls[0])
            if cls < 0 or cls >= len(class_names):
                continue
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            label = class_names[cls]
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(frame, label, (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    cv2.imshow("Video", frame)

    if cv2.waitKey(25) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

Objek car disimpan ke detections\car_0001.jpg
Objek kaca disimpan ke detections\kaca_0002.jpg
Objek driver_buckled disimpan ke detections\driver_buckled_0003.jpg
Objek motor_1_helmet disimpan ke detections\motor_1_helmet_0004.jpg
Objek motor_2_helmet disimpan ke detections\motor_2_helmet_0005.jpg
Objek car disimpan ke detections\car_0006.jpg
Objek kaca disimpan ke detections\kaca_0007.jpg
Objek motor_1_helmet disimpan ke detections\motor_1_helmet_0008.jpg
Objek motor_2_nohelmet disimpan ke detections\motor_2_nohelmet_0009.jpg
Objek plat_nomor disimpan ke detections\plat_nomor_0010.jpg
Objek driver_unknown disimpan ke detections\driver_unknown_0011.jpg
Objek motor_2_helmet disimpan ke detections\motor_2_helmet_0012.jpg
Objek car disimpan ke detections\car_0013.jpg
Objek motor_1_helmet disimpan ke detections\motor_1_helmet_0014.jpg
Objek kaca disimpan ke detections\kaca_0015.jpg
Objek plat_nomor disimpan ke detections\plat_nomor_0016.jpg
Objek motor_2_helmet disimpan ke detections\motor_

In [38]:
import cv2
import os
from ultralytics import YOLO

# Load model hasil training
model = YOLO(r"C:\Users\EagleEyes\belajar\model\best94.pt")

# Buka file video
cap = cv2.VideoCapture(r'C:\Users\EagleEyes\dataset\video\jpo_lenteng_agung2.mp4')

# Daftar nama class sesuai training
class_names = [
    'car', 
    'driver_buckled', 
    'driver_unbuckled', 
    'driver_unknown', 
    'kaca',
    'motor_1_helmet', 
    'motor_1_nohelmet', 
    'motor_2_helmet', 
    'motor_2_nohelmet',
    'motor_more_2', 
    'passanger_buckled', 
    'passanger_unbuckled', 
    'passanger_unknown',
    'plat_nomor'
]

# Buat direktori output utama
output_dir = "detections"
os.makedirs(output_dir, exist_ok=True)

frame_count = 0
save_counts = {cls: 0 for cls in class_names}  # counter per class

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1
    results = []  # default kosong → biar aman kalau frame tidak diproses

    # Jalankan deteksi setiap 20 frame
    if frame_count % 20 == 0:
        results = model(frame, verbose=False)

        for r in results:
            boxes = r.boxes
            if len(boxes) > 0:
                for box in boxes:
                    cls = int(box.cls[0])
                    conf = float(box.conf[0])

                    # Pastikan index valid
                    if cls < 0 or cls >= len(class_names):
                        continue

                    # Koordinat bounding box
                    x1, y1, x2, y2 = map(int, box.xyxy[0])

                    # Crop objek dari frame
                    cropped = frame[y1:y2, x1:x2]

                    if cropped.size == 0:
                        continue  # skip kalau crop kosong

                    # Simpan sesuai nama class → di folder khusus
                    class_name = class_names[cls]
                    class_dir = os.path.join(output_dir, class_name)
                    os.makedirs(class_dir, exist_ok=True)

                    save_counts[class_name] += 1
                    save_path = os.path.join(
                        class_dir,
                        f"{class_name}_{save_counts[class_name]:04d}.jpg"
                    )
                    cv2.imwrite(save_path, cropped)
                    print(f"Objek {class_name} disimpan ke {save_path}")

    # --- tampilkan video dengan box hanya kalau ada deteksi ---
    if results:
        for r in results:
            for box in r.boxes:
                cls = int(box.cls[0])
                if cls < 0 or cls >= len(class_names):
                    continue
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                label = class_names[cls]
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(frame, label, (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    # Resize frame sebelum ditampilkan (800x600)
    resized_frame = cv2.resize(frame, (1024, 800))
    cv2.imshow("Video", resized_frame)

    if cv2.waitKey(25) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()


Objek car disimpan ke detections\car\car_0001.jpg
Objek kaca disimpan ke detections\kaca\kaca_0001.jpg
Objek driver_buckled disimpan ke detections\driver_buckled\driver_buckled_0001.jpg
Objek motor_1_helmet disimpan ke detections\motor_1_helmet\motor_1_helmet_0001.jpg
Objek motor_2_helmet disimpan ke detections\motor_2_helmet\motor_2_helmet_0001.jpg
Objek car disimpan ke detections\car\car_0002.jpg
Objek kaca disimpan ke detections\kaca\kaca_0002.jpg
Objek motor_1_helmet disimpan ke detections\motor_1_helmet\motor_1_helmet_0002.jpg
Objek motor_2_nohelmet disimpan ke detections\motor_2_nohelmet\motor_2_nohelmet_0001.jpg
Objek plat_nomor disimpan ke detections\plat_nomor\plat_nomor_0001.jpg
Objek driver_unknown disimpan ke detections\driver_unknown\driver_unknown_0001.jpg
Objek motor_2_helmet disimpan ke detections\motor_2_helmet\motor_2_helmet_0002.jpg
Objek car disimpan ke detections\car\car_0003.jpg
Objek motor_1_helmet disimpan ke detections\motor_1_helmet\motor_1_helmet_0003.jpg
Obj

In [31]:
import cv2
import os
from ultralytics import YOLO

# Load model hasil training
model = YOLO(r"C:\Users\EagleEyes\belajar\model\best94.pt")

# Buka file video
name_file =  "jpo_lenteng_agung2.mp4"
cap = cv2.VideoCapture(f"video/{name_file}")

# Daftar nama class sesuai training
class_names = [
    'car', 
    'driver_buckled', 
    'driver_unbuckled', 
    'driver_unknown', 
    'kaca',
    'motor_1_helmet', 
    'motor_1_nohelmet', 
    'motor_2_helmet', 
    'motor_2_nohelmet',
    'motor_more_2', 
    'passanger_buckled', 
    'passanger_unbuckled', 
    'passanger_unknown',
    'plat_nomor'
]

# Buat direktori output
output_dir = "detections"
frames_dir = os.path.join(output_dir, "frames")
frames_dir_result = os.path.join(output_dir, "frames_result")
os.makedirs(output_dir, exist_ok=True)
os.makedirs(frames_dir, exist_ok=True)
os.makedirs(frames_dir_result, exist_ok=True)

frame_count = 0
save_counts = {cls: 0 for cls in class_names}  # counter per class
frame_save_count = 0  # counter frame utuh

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1
    results = []  # default kosong → biar aman kalau frame tidak diproses

    # Jalankan deteksi setiap 30 frame
    if frame_count % 10 == 0:
        # Simpan frame utuh
        frame_save_count += 1
        frame_path = os.path.join(frames_dir, f"frame_{name_file}_{frame_save_count:04d}.jpg")
        cv2.imwrite(frame_path, frame)
        print(f"Frame utuh disimpan ke {frame_path}")

        # Jalankan deteksi YOLO
        results = model(frame, verbose=False)

        for r in results:
            boxes = r.boxes
            if len(boxes) > 0:
                for box in boxes:
                    cls = int(box.cls[0])
                    conf = float(box.conf[0])

                    # Pastikan index valid
                    if cls < 0 or cls >= len(class_names):
                        continue

                    # Koordinat bounding box
                    x1, y1, x2, y2 = map(int, box.xyxy[0])

                    # Crop objek dari frame
                    cropped = frame[y1:y2, x1:x2]

                    if cropped.size == 0:
                        continue  # skip kalau crop kosong

                    # Simpan sesuai nama class → di folder khusus
                    class_name = class_names[cls]
                    class_dir = os.path.join(output_dir, class_name)
                    os.makedirs(class_dir, exist_ok=True)

                    save_counts[class_name] += 1
                    save_path = os.path.join(
                        class_dir,
                        f"{class_name}_{save_counts[class_name]:04d}.jpg"
                    )
                    cv2.imwrite(save_path, cropped)
                    print(f"Objek {class_name} disimpan ke {save_path}")

    # --- tampilkan video dengan box hanya kalau ada deteksi ---
    if results:
        for r in results:
            for box in r.boxes:
                cls = int(box.cls[0])
                if cls < 0 or cls >= len(class_names):
                    continue
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                label = class_names[cls]
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(frame, label, (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        
        # Simpan frame result
        frame_path_result = os.path.join(frames_dir_result, f"frame_{name_file}_{frame_save_count:04d}_result.jpg")
        cv2.imwrite(frame_path_result, frame)
        print(f"Frame result  disimpan ke {frame_path_result} ")

    # Resize frame jadi setengah ukuran asli
    resized_frame = cv2.resize(frame, (0, 0), fx=0.5, fy=0.5)
    cv2.imshow("Video", resized_frame)

    if cv2.waitKey(25) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()


Frame utuh disimpan ke detections\frames\frame_jpo_pondok_bambu.mp4_0001.jpg
Frame result  disimpan ke detections\frames_result\frame_jpo_pondok_bambu.mp4_0001_result.jpg 
Frame utuh disimpan ke detections\frames\frame_jpo_pondok_bambu.mp4_0002.jpg
Frame result  disimpan ke detections\frames_result\frame_jpo_pondok_bambu.mp4_0002_result.jpg 
Frame utuh disimpan ke detections\frames\frame_jpo_pondok_bambu.mp4_0003.jpg
Frame result  disimpan ke detections\frames_result\frame_jpo_pondok_bambu.mp4_0003_result.jpg 
Frame utuh disimpan ke detections\frames\frame_jpo_pondok_bambu.mp4_0004.jpg
Objek motor_1_helmet disimpan ke detections\motor_1_helmet\motor_1_helmet_0001.jpg
Frame result  disimpan ke detections\frames_result\frame_jpo_pondok_bambu.mp4_0004_result.jpg 
Frame utuh disimpan ke detections\frames\frame_jpo_pondok_bambu.mp4_0005.jpg
Objek motor_1_helmet disimpan ke detections\motor_1_helmet\motor_1_helmet_0002.jpg
Objek plat_nomor disimpan ke detections\plat_nomor\plat_nomor_0001.jp

In [39]:
import cv2
import os
from ultralytics import YOLO

# Load model hasil training
model = YOLO(r"C:\Users\EagleEyes\belajar\model\best94.pt")

# Buka file video
cap = cv2.VideoCapture(r"C:\Users\EagleEyes\belajar\video\jpo_lenteng_agung2.mp4")

# Daftar nama class sesuai training
class_names = [
    'car', 
    'driver_buckled', 
    'driver_unbuckled', 
    'driver_unknown', 
    'kaca',
    'motor_1_helmet', 
    'motor_1_nohelmet', 
    'motor_2_helmet', 
    'motor_2_nohelmet',
    'motor_more_2', 
    'passanger_buckled', 
    'passanger_unbuckled', 
    'passanger_unknown',
    'plat_nomor'
]

# Buat direktori output utama
output_dir = "detections"
os.makedirs(output_dir, exist_ok=True)

frame_count = 0
save_counts = {cls: 0 for cls in class_names}  # counter per class

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1
    results = []  # default kosong → biar aman kalau frame tidak diproses

    # Jalankan deteksi setiap 20 frame
    if frame_count % 20 == 0:
        results = model(frame, verbose=False)

        for r in results:
            boxes = r.boxes
            if len(boxes) > 0:
                for box in boxes:
                    cls = int(box.cls[0])
                    conf = float(box.conf[0])

                    # Pastikan index valid
                    if cls < 0 or cls >= len(class_names):
                        continue

                    # Koordinat bounding box
                    x1, y1, x2, y2 = map(int, box.xyxy[0])

                    # Crop objek dari frame
                    cropped = frame[y1:y2, x1:x2]

                    if cropped.size == 0:
                        continue  # skip kalau crop kosong

                    # Simpan sesuai nama class → di folder khusus
                    class_name = class_names[cls]
                    class_dir = os.path.join(output_dir, class_name)
                    os.makedirs(class_dir, exist_ok=True)

                    save_counts[class_name] += 1
                    save_path = os.path.join(class_dir, f"{class_name}_{save_counts[class_name]:04d}.jpg")
                    cv2.imwrite(save_path, cropped)
                    print(f"Objek {class_name} disimpan ke {save_path}")

    # --- tampilkan video dengan box hanya kalau ada deteksi ---
    if results:
        for r in results:
            for box in r.boxes:
                cls = int(box.cls[0])
                if cls < 0 or cls >= len(class_names):
                    continue
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                label = class_names[cls]
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(frame, label, (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    cv2.imshow("Video", frame)

    if cv2.waitKey(25) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()


Objek car disimpan ke detections\car\car_0001.jpg
Objek kaca disimpan ke detections\kaca\kaca_0001.jpg
Objek driver_buckled disimpan ke detections\driver_buckled\driver_buckled_0001.jpg
Objek motor_1_helmet disimpan ke detections\motor_1_helmet\motor_1_helmet_0001.jpg
Objek motor_2_helmet disimpan ke detections\motor_2_helmet\motor_2_helmet_0001.jpg
Objek car disimpan ke detections\car\car_0002.jpg
Objek kaca disimpan ke detections\kaca\kaca_0002.jpg
Objek motor_1_helmet disimpan ke detections\motor_1_helmet\motor_1_helmet_0002.jpg
Objek motor_2_nohelmet disimpan ke detections\motor_2_nohelmet\motor_2_nohelmet_0001.jpg
Objek plat_nomor disimpan ke detections\plat_nomor\plat_nomor_0001.jpg
Objek driver_unknown disimpan ke detections\driver_unknown\driver_unknown_0001.jpg
Objek motor_2_helmet disimpan ke detections\motor_2_helmet\motor_2_helmet_0002.jpg
Objek car disimpan ke detections\car\car_0003.jpg
Objek motor_1_helmet disimpan ke detections\motor_1_helmet\motor_1_helmet_0003.jpg
Obj

Cek Berpasangan JPG dan XML

In [1]:
import os

# Path ke folder yang berisi file foto dan XML
folder_path = r"C:\Users\EagleEyes\belajar\dataset_mentah"
    
# Pastikan folder ada
if not os.path.exists(folder_path):
    print(f"Error: Folder {folder_path} tidak ditemukan.")
    exit()

# Dapatkan daftar file di folder
files = os.listdir(folder_path)

# Pisahkan file berdasarkan ekstensi
jpg_files = [f for f in files if f.lower().endswith('.jpg')]
xml_files = [f for f in files if f.lower().endswith('.xml')]

# Dapatkan nama file tanpa ekstensi
jpg_basenames = {os.path.splitext(f)[0] for f in jpg_files}
xml_basenames = {os.path.splitext(f)[0] for f in xml_files}

# Temukan file yang tidak berpasangan
jpg_without_xml = jpg_basenames - xml_basenames  # JPG tanpa XML
xml_without_jpg = xml_basenames - jpg_basenames  # XML tanpa JPG

# Gabungkan file yang akan dihapus
files_to_delete = [f"{name}.jpg" for name in jpg_without_xml] + [f"{name}.xml" for name in xml_without_jpg]

# Hapus file yang tidak berpasangan
if not files_to_delete:
    print("Tidak ada file yang tidak berpasangan ditemukan.")
else:
    print("File yang akan dihapus:")
    for file in files_to_delete:
        file_path = os.path.join(folder_path, file)
        try:
            os.remove(file_path)
            print(f"Berhasil menghapus: {file_path}")
        except Exception as e:
            print(f"Gagal menghapus {file_path}: {e}")

Tidak ada file yang tidak berpasangan ditemukan.


Cek Jumlah Class

In [2]:
import os
import xml.etree.ElementTree as ET

folder_path = r"C:\Users\EagleEyes\belajar\dataset_mentah"
label_set = set()

# Iterasi semua file XML
for filename in os.listdir(folder_path):
    if filename.endswith(".xml"):
        file_path = os.path.join(folder_path, filename)
        try:
            tree = ET.parse(file_path)
            root = tree.getroot()
            for obj in root.findall('object'):
                label = obj.find('name').text
                if label:
                    label_set.add(label)
        except ET.ParseError:
            print(f"Gagal parsing: {filename}")

# Tampilkan hasil
print(f"Jumlah label unik: {len(label_set)}")
print("Label-label yang ditemukan:")
for label in sorted(label_set):
    print(f"- {label}")

Jumlah label unik: 14
Label-label yang ditemukan:
- car
- driver_buckled
- driver_unbuckled
- driver_unknown
- kaca
- motor_1_helmet
- motor_1_nohelmet
- motor_2_helmet
- motor_2_nohelmet
- motor_more_2
- passanger_buckled
- passanger_unbuckled
- passanger_unknown
- plat_nomor


Split Data

In [3]:
import os
import shutil
import random
import xml.etree.ElementTree as ET

# Path ke folder sumber dan tujuan
source_folder = r"C:\Users\EagleEyes\belajar\dataset_mentah"
train_img_folder = r"C:\Users\EagleEyes\belajar\dataset\images\train"
val_img_folder = r"C:\Users\EagleEyes\belajar\dataset\images\val"
# test_img_folder = r"C:\Users\EagleEyes\belajar\dataset\images\test"
train_xml_folder = r"C:\Users\EagleEyes\belajar\dataset\annotations\train"
val_xml_folder = r"C:\Users\EagleEyes\belajar\dataset\annotations\val"
# test_xml_folder = r"C:\Users\EagleEyes\belajar\dataset\annotations\test"

# Pastikan folder sumber ada
if not os.path.exists(source_folder):
    print(f"Error: Folder {source_folder} tidak ditemukan.")
    exit()

# Buat folder tujuan jika belum ada
for folder in [train_img_folder, val_img_folder, train_xml_folder, val_xml_folder]:
    os.makedirs(folder, exist_ok=True)

# Dapatkan daftar file di folder sumber
files = os.listdir(source_folder)
jpg_files = [f for f in files if f.lower().endswith('.jpg')]
xml_files = [f for f in files if f.lower().endswith('.xml')]

# Dapatkan nama file tanpa ekstensi yang berpasangan
jpg_basenames = {os.path.splitext(f)[0] for f in jpg_files}
xml_basenames = {os.path.splitext(f)[0] for f in xml_files}
paired_basenames = jpg_basenames.intersection(xml_basenames)  # Hanya file yang berpasangan

# Konversi ke list dan acak untuk pembagian
paired_files = list(paired_basenames)
random.shuffle(paired_files)

# Hitung jumlah file untuk train (80%) dan val (20%)
total_files = len(paired_files)
train_count = int(total_files * 0.8)
val_count = total_files - train_count  # Sisanya untuk val

# Bagi file menjadi train dan val
train_files = paired_files[:train_count]
val_files = paired_files[train_count:]

# Fungsi untuk mengubah path dalam file XML
def update_xml_path(xml_file, new_img_path):
    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()
        path_element = root.find('path')
        if path_element is not None:
            path_element.text = new_img_path
        tree.write(xml_file)
    except Exception as e:
        print(f"Gagal memproses XML {xml_file}: {e}")

# Salin file dan perbarui XML
def copy_files(file_list, img_dest_folder, xml_dest_folder, source_folder):
    for basename in file_list:
        jpg_file = f"{basename}.jpg"
        xml_file = f"{basename}.xml"
        src_jpg_path = os.path.join(source_folder, jpg_file)
        src_xml_path = os.path.join(source_folder, xml_file)
        dest_jpg_path = os.path.join(img_dest_folder, jpg_file)
        dest_xml_path = os.path.join(xml_dest_folder, xml_file)

        # Salin file JPG
        try:
            shutil.copy2(src_jpg_path, dest_jpg_path)
            print(f"Berhasil menyalin {jpg_file} ke {img_dest_folder}")
        except Exception as e:
            print(f"Gagal menyalin {jpg_file}: {e}")

        # Salin file XML dan perbarui path
        try:
            shutil.copy2(src_xml_path, dest_xml_path)
            update_xml_path(dest_xml_path, dest_jpg_path)
            print(f"Berhasil menyalin dan memperbarui {xml_file} ke {xml_dest_folder}")
        except Exception as e:
            print(f"Gagal menyalin atau memperbarui {xml_file}: {e}")

# Salin 80% ke train
copy_files(train_files, train_img_folder, train_xml_folder, source_folder)

# Salin 20% ke val
copy_files(val_files, val_img_folder, val_xml_folder, source_folder)

# Salin 100% ke test
#copy_files(paired_files, test_img_folder, test_xml_folder, source_folder)

print("Proses penyalinan dan pembaruan selesai.")

Berhasil menyalin frame_cam1.mp4_0088.jpg ke C:\Users\EagleEyes\belajar\dataset\images\train
Berhasil menyalin dan memperbarui frame_cam1.mp4_0088.xml ke C:\Users\EagleEyes\belajar\dataset\annotations\train
Berhasil menyalin frame_2.mov_0080.jpg ke C:\Users\EagleEyes\belajar\dataset\images\train
Berhasil menyalin dan memperbarui frame_2.mov_0080.xml ke C:\Users\EagleEyes\belajar\dataset\annotations\train
Berhasil menyalin frame_cam2.mp4_0068.jpg ke C:\Users\EagleEyes\belajar\dataset\images\train
Berhasil menyalin dan memperbarui frame_cam2.mp4_0068.xml ke C:\Users\EagleEyes\belajar\dataset\annotations\train
Berhasil menyalin frame_cam1.mp4_0027.jpg ke C:\Users\EagleEyes\belajar\dataset\images\train
Berhasil menyalin dan memperbarui frame_cam1.mp4_0027.xml ke C:\Users\EagleEyes\belajar\dataset\annotations\train
Berhasil menyalin 20250329_133605_frame00760.jpg ke C:\Users\EagleEyes\belajar\dataset\images\train
Berhasil menyalin dan memperbarui 20250329_133605_frame00760.xml ke C:\Users\E

Convert xml to txt

In [4]:
import os
import xml.etree.ElementTree as ET

# === SETTING ===
dataset_dir = "dataset/annotations"  # folder utama
splits = ["train", "val"]

# mapping kelas
class_mapping = {
    "car": 0,
    "driver_buckled": 1,
    "driver_unbuckled": 2,
    "driver_unknown": 3,
    "kaca": 4,
    "motor_1_helmet": 5,
    "motor_1_nohelmet": 6,
    "motor_2_helmet": 7,
    "motor_2_nohelmet": 8,
    "motor_more_2": 9,
    "passanger_buckled": 10,
    "passanger_unbuckled": 11,
    "passanger_unknown": 12,
    "plat_nomor": 13
}

# simpan daftar kelas ke file classes.txt
classes_file = os.path.join(dataset_dir, "classes.txt")
with open(classes_file, "w", encoding="utf-8") as f:
    for cls in class_mapping.keys():
        f.write(cls + "\n")
print(f"[INFO] classes.txt berhasil dibuat di {classes_file}")

def convert_voc_to_yolo(xml_file, label_file, class_mapping):
    tree = ET.parse(xml_file)
    root = tree.getroot()

    size = root.find("size")
    if size is None:
        return

    img_w = float(size.find("width").text)
    img_h = float(size.find("height").text)

    lines = []
    for obj in root.findall(".//object"):
        cls_name = obj.find("name").text.strip()
        if cls_name not in class_mapping:
            continue

        cls_id = class_mapping[cls_name]
        bndbox = obj.find("bndbox")
        xmin = float(bndbox.find("xmin").text)
        ymin = float(bndbox.find("ymin").text)
        xmax = float(bndbox.find("xmax").text)
        ymax = float(bndbox.find("ymax").text)

        # YOLO format
        x_center = (xmin + xmax) / 2.0 / img_w
        y_center = (ymin + ymax) / 2.0 / img_h
        bw = (xmax - xmin) / img_w
        bh = (ymax - ymin) / img_h

        lines.append(f"{cls_id} {x_center:.6f} {y_center:.6f} {bw:.6f} {bh:.6f}")

    if lines:
        with open(label_file, "w", encoding="utf-8") as f:
            f.write("\n".join(lines))

# === PROSES SEMUA SPLIT ===
for split in splits:
    ann_dir = os.path.join(dataset_dir, split)   # xml ada di sini
    label_dir = os.path.join(dataset_dir, "../labels", split)
    os.makedirs(label_dir, exist_ok=True)

    for fname in os.listdir(ann_dir):
        if not fname.lower().endswith(".xml"):
            continue

        xml_file = os.path.join(ann_dir, fname)
        txt_file = os.path.join(label_dir, os.path.splitext(fname)[0] + ".txt")

        try:
            convert_voc_to_yolo(xml_file, txt_file, class_mapping)
            print(f"[OK] {split}/{fname} -> labels/{split}/{os.path.basename(txt_file)}")
        except Exception as e:
            print(f"[WARNING] Gagal proses {fname}: {e}")

print("[DONE] Semua XML berhasil dikonversi sesuai struktur dataset/")


[INFO] classes.txt berhasil dibuat di dataset/annotations\classes.txt
[OK] train/20250329_132521_frame00020.xml -> labels/train/20250329_132521_frame00020.txt
[OK] train/20250329_132521_frame00220.xml -> labels/train/20250329_132521_frame00220.txt
[OK] train/20250329_132521_frame00280.xml -> labels/train/20250329_132521_frame00280.txt
[OK] train/20250329_132521_frame00300.xml -> labels/train/20250329_132521_frame00300.txt
[OK] train/20250329_132521_frame00320.xml -> labels/train/20250329_132521_frame00320.txt
[OK] train/20250329_132521_frame00340.xml -> labels/train/20250329_132521_frame00340.txt
[OK] train/20250329_132521_frame00360.xml -> labels/train/20250329_132521_frame00360.txt
[OK] train/20250329_132521_frame00400.xml -> labels/train/20250329_132521_frame00400.txt
[OK] train/20250329_132521_frame00420.xml -> labels/train/20250329_132521_frame00420.txt
[OK] train/20250329_132521_frame00500.xml -> labels/train/20250329_132521_frame00500.txt
[OK] train/20250329_132521_frame00520.xm

In [ ]:
import torch
print(f"GPU tersedia: {torch.cuda.is_available()}")
print(f"Nama GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Tidak ada GPU'}")

Train Epoch

In [ ]:
# Epoch 100
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # atau yolov8s.pt

results = model.train(
    data="C:/Users/EagleEyes/belajar/data.yaml",
    epochs=100,
    batch=16,
    #lr0=0.001,
    #lrf=0.001,
    #device=0,
    project="C:/Users/EagleEyes/belajar/runs"
)

In [5]:
# Epoch 500
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

# training
results = model.train(
    data="C:/Users/EagleEyes/belajar/data.yaml",
    epochs=500,
    batch=16,
    imgsz=640,
    patience=0,   # <-- disable early stopping, biar jalan full 500 epoch
    project="C:/Users/EagleEyes/belajar/runs"
)

C:\Users\EagleEyes\anaconda3\envs\yolov8\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


New https://pypi.org/project/ultralytics/8.3.199 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.197  Python-3.10.18 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060, 12288MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:/Users/EagleEyes/belajar/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train5, nbs=64